# DeepEval Multi-Turn Evaluation: Custom Binary Metrics from Domain Failure Modes

## Overview

This notebook demonstrates how to evaluate multi-turn conversations with [DeepEval](https://deepeval.com/guides/guides-ai-agent-evaluation), using **custom binary metrics built from the travel booking agent's real failure modes** rather than off-the-shelf numeric metrics.

The pattern we follow here is the same one used in notebook 05 with Strands Evals:

1. Enumerate concrete failure modes of the agent.
2. Write one binary pass/fail criterion per failure mode.
3. Express each criterion as a `ConversationalGEval` (single-criterion LLM judge) or a `ConversationalDAGMetric` (multi-step decision tree).
4. Read results as **pass rate per failure mode**, not as aggregate Likert scores.

## What You'll Learn

1. How to use `ConversationalGolden` and `ConversationSimulator` to generate the test cases we'll score (quick recap from notebook 03).
2. How to wire Amazon Nova Micro as the simulator and Claude Sonnet 4.5 as the evaluation judge.
3. How to define **custom binary** multi-turn metrics with `ConversationalGEval`, one per failure mode.
4. How to build a **custom decision-tree** metric with `ConversationalDAGMetric` for composite checks that only make sense in a specific order.
5. How to aggregate results as a pass-rate-per-failure-mode table.


## Key Components

- **ConversationalGolden** - A test specification: scenario, user persona, expected outcome. No scripted messages.
- **ConversationSimulator** - Generates realistic user messages turn by turn and feeds them to your application.
- **ConversationalTestCase** - The resulting conversation, ready to score.
- **ConversationalGEval** - Applies a single natural-language criterion (phrased as a pass/fail question) to a whole conversation.
- **ConversationalDAGMetric** - Traverses a decision tree so later checks only run when earlier ones pass.

In [ ]:
# Install all required packages (deepeval, strands, boto3, etc.)
%pip install -r requirements.txt

The following code imports the required agent and tool components from Strands framework.

In [ ]:
# Import Strands Agent class and tool decorator for defining agent tools
from strands import Agent, tool
from datetime import date

Now, let's define the assistant's code. The cell below defines the in-memory booking state and six tools that the agent will use: `search_flights`, `search_hotels`, `book_flight`, `book_hotel`, `get_bookings`, and `cancel_booking`. Each tool returns a formatted string that the agent relays to the user.

In [ ]:
# --- In-memory booking state ---
bookings: dict = {}
booking_counter = 0


# --- Tool definitions: each @tool becomes available to the agent ---

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Search for available flights between two cities.

    Args:
        origin: Departure city or airport code (e.g. 'New York' or 'JFK')
        destination: Arrival city or airport code (e.g. 'London' or 'LHR')
        departure_date: Departure date in YYYY-MM-DD format
        return_date: Optional return date in YYYY-MM-DD format for round trips
    """
    flights = [
        {"flight": "AA101", "airline": "American Airlines", "departs": "08:00", "arrives": "20:00", "price": 450, "class": "Economy"},
        {"flight": "BA202", "airline": "British Airways", "departs": "11:30", "arrives": "23:45", "price": 620, "class": "Economy"},
        {"flight": "UA303", "airline": "United Airlines", "departs": "14:00", "arrives": "02:15", "price": 890, "class": "Business"},
    ]
    trip = f"round-trip (return: {return_date})" if return_date else "one-way"
    lines = [f"Flights from {origin} to {destination} on {departure_date} ({trip}):\n"]
    for f in flights:
        lines.append(
            f" {f['flight']} | {f['airline']} | {f['departs']}-{f['arrives']} | "
            f"${f['price']} | {f['class']}"
        )
    return "\n".join(lines)


@tool
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Search for available hotels in a city.

    Args:
        city: Destination city
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
    """
    hotels = [
        {"name": "Grand Plaza Hotel", "stars": 5, "price_per_night": 320, "amenities": "Pool, Spa, Restaurant"},
        {"name": "City Center Inn", "stars": 3, "price_per_night": 95, "amenities": "Free WiFi, Breakfast"},
        {"name": "Boutique Stay", "stars": 4, "price_per_night": 180, "amenities": "Gym, Bar, Concierge"},
    ]
    nights = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    lines = [f"Hotels in {city} | {check_in} → {check_out} ({nights} nights, {guests} guest(s)):\n"]
    for h in hotels:
        total = h["price_per_night"] * nights
        lines.append(
            f" {'★' * h['stars']} {h['name']} | ${h['price_per_night']}/night (total ${total}) | {h['amenities']}"
        )
    return "\n".join(lines)


@tool
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> str:
    """Book a flight for a passenger.

    Args:
        flight_number: Flight number to book (e.g. 'AA101')
        passenger_name: Full name of the passenger
        origin: Departure city or airport
        destination: Arrival city or airport
        travel_date: Travel date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"FLT-{booking_counter:04d}"
    bookings[ref] = {
        "type": "flight", "flight": flight_number, "passenger": passenger_name,
        "route": f"{origin} → {destination}", "date": travel_date, "status": "Confirmed"
    }
    return f"✅ Flight booked! Ref: {ref} | {flight_number} | {origin} → {destination} on {travel_date} | Passenger: {passenger_name}"


@tool
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.

    Args:
        hotel_name: Name of the hotel to book
        guest_name: Full name of the guest
        city: City where the hotel is located
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"HTL-{booking_counter:04d}"
    bookings[ref] = {
        "type": "hotel", "hotel": hotel_name, "guest": guest_name,
        "city": city, "check_in": check_in, "check_out": check_out, "status": "Confirmed"
    }
    return f"✅ Hotel booked! Ref: {ref} | {hotel_name} in {city} | {check_in} → {check_out} | Guest: {guest_name}"


@tool
def get_bookings() -> str:
    """Retrieve all current bookings."""
    if not bookings:
        return "No bookings found."
    lines = ["Current bookings:\n"]
    for ref, b in bookings.items():
        if b["type"] == "flight":
            lines.append(f" [{ref}] ✈ {b['flight']} | {b['route']} on {b['date']} | {b['passenger']} | {b['status']}")
        else:
            lines.append(f" [{ref}] 🏨 {b['hotel']} in {b['city']} | {b['check_in']} → {b['check_out']} | {b['guest']} | {b['status']}")
    return "\n".join(lines)


@tool
def cancel_booking(booking_ref: str) -> str:
    """Cancel an existing booking.

    Args:
        booking_ref: Booking reference number (e.g. 'FLT-0001' or 'HTL-0002')
    """
    if booking_ref not in bookings:
        return f"Booking {booking_ref} not found."
    bookings[booking_ref]["status"] = "Cancelled"
    return f"❌ Booking {booking_ref} has been cancelled."

# All tools return formatted strings so the agent can relay results to the user.

Now we initialize the Strands agent with a travel-assistant system prompt and all six tools defined above.

In [ ]:
# Initialize the travel-assistant agent with a system prompt and all booking tools
agent = Agent(
    system_prompt=(
        "You are a helpful travel booking assistant. You help users search for flights and hotels, "
        "make bookings, view existing reservations, and cancel bookings. "
        "Always confirm details with the user before completing a booking. "
        "Use today's date as context when dates are not fully specified."
    ),
    tools=[search_flights, search_hotels, book_flight, book_hotel, get_bookings, cancel_booking],
    callback_handler=None, # suppress streaming output to avoid Rich/Jupyter recursion
)

## Quick Demo: Agent Invocation

Before diving into evaluation, let's verify the agent works with a single-turn request that searches for flights and books the cheapest option.

In [ ]:
# Quick smoke test: search flights and book the cheapest in a single turn
agent("Search for flights from New York to London on 2025-09-01, then book the cheapest one for Jane Doe")

# DeepEval Multi-Turn Evaluation Framework

Every change in your GenAI application, whether it is a prompt revision, a model swap, or a retrieval pipeline update, carries the risk of improving one conversational behavior while quietly degrading another. Multi-turn evaluation gives you a controlled way to measure that tradeoff. By running successive versions of your application through the same set of scenarios, you can quantify exactly where a new release improves, where it regresses, and where it holds steady, before any of those changes reach production. DeepEval's multi-turn evaluation framework helps to implement this mechanism.

DeepEval's multi-turn evaluation framework is built around three components that comprise the evaluation flow: *ConversationalGolden, ConversationSimulator*, and *ConversationalTestCase*. ConversationalGolden defines what to evaluate, ConversationSimulator simulates the conversation and applies the evaluation scenarios to your GenAI application, ConversationalTestCase captures the results of these runs for evaluation scoring.

The simulation of scenarios follows a three-stage loop that mirrors how a real user would interact with your application:

1. **Scenario initialization**. A ConversationalGolden supplies the ConversationSimulator with everything it needs to begin: the scenario description, the simulated user's persona, and the expected outcome that signals a successful resolution.
2. **Turn-by-turn generation**. The simulator produces a realistic user message based on the persona and conversation history so far, then passes it to your GenAI application, which responds as it would in production. This back-and-forth continues iteratively until either the conversation reaches the defined expected outcome or the maximum turn limit is hit.
3. **Packaging for evaluation**. Once the conversation concludes, the entire exchange, every user message and assistant reply in sequence, is bundled into a ConversationalTestCase, ready to be scored against your multi-turn metrics.

## Build Golden Scenarios with ConversationalGolden

Rather than defining a fixed script of messages and expected replies, a *ConversationalGolden* captures the conditions under which a conversation should take place. It includes a natural-language scenario description (e.g., "A budget-conscious couple planning a weekend getaway to Paris"), a persona that defines how the simulated user should behave, and an expected outcome that describes what a successful resolution looks like. 

Think of a *ConversationalGolden* as the test specification: it tells the system what to test without prescribing how the conversation will unfold.

*ConversationalGolden* fields are:

| Field | Purpose |
|---|---|
| `scenario` | Describes the conversational situation to simulate |
| `expected_outcome` | Defines what a successful resolution looks like |
| `user_description` | Sets the persona and behavioral traits of the simulated user |

*ConversationGolden* items comprise the *EvaluationDataset* that can be treated as a code, versioned, and stored in a source code control tool.

The next snippet of code defines two `ConversationalGolden` scenarios for two different customer personas. 

In [ ]:
from deepeval.dataset import ConversationalGolden

# Define two evaluation scenarios with distinct user personas
goldens=[
    ConversationalGolden(
        scenario="Solo traveler searching for a cheap one-way flight from New York to London and booking the most affordable option",
        expected_outcome="User finds a suitable flight and receives a confirmed booking reference",
        user_description="Budget-conscious traveler who prioritizes low cost over comfort or schedule"
    ), 
    ConversationalGolden(
        scenario="User who already has multiple bookings and wants to review them before deciding which one to cancel",
        expected_outcome="User retrieves all current bookings, selects one to cancel, and verifies the remaining bookings are still intact",
        user_description="Cautious user who double-checks details and asks for confirmation before making any changes"
    ) 
]

print(f"Defined {len(goldens)} conversation scenarios.")

The quality of your evaluation is directly determined by the quality of the scenarios driving it. 

Target at least 30 scenarios, spread deliberately across core workflows that cover everyday usage, edge cases that probe the boundaries of your agent's capabilities, and adversarial situations where conversations are most likely to break down. 

Vary user behavior by pairing identical scenarios with distinct personas to test adaptability.

In [ ]:
# Same scenario paired with two contrasting personas to test adaptability

# Relaxed, flexible traveler
goldens.append(ConversationalGolden(
    scenario="Traveler searching for a round-trip flight from Boston to Rome",
    expected_outcome="User compares options and books a flight",
    user_description="Easygoing traveler with flexible dates who is happy to take recommendations"
))

# Demanding, detail-oriented traveler
goldens.append(ConversationalGolden(
    scenario="Traveler searching for a round-trip flight from Boston to Rome",
    expected_outcome="User compares options and books a flight",
    user_description="Meticulous planner who asks about every fare detail, baggage policy, and layover duration before committing"
))


Target edge cases explicitly rather than relying on organic variation to surface boundary conditions, and test multi-topic conversations in which users switch between tasks mid-session, as these are where context retention failures are most likely to emerge.

In [ ]:
# Edge case: invalid (past) date handling
goldens.append(ConversationalGolden(
    scenario="User tries to book a flight for a date that has already passed",
    expected_outcome="Agent recognizes the invalid date and prompts the user to provide a future date",
    user_description="Distracted user who does not realize they entered the wrong date"
))

# Multi-topic conversation: flight + hotel booking then partial cancellation
goldens.append(ConversationalGolden(
    scenario="User searches for flights to Tokyo, then pivots to finding a hotel, books both, and finally asks to cancel just the hotel",
    expected_outcome="User ends with a confirmed flight booking and a successfully canceled hotel booking",
    user_description="Decisive but unpredictable user whose plans shift as the conversation progresses"
))


## 2.2 Simulate User Conversation with ConversationSimulator

*ConversationSimulator* reads the scenario and persona from a *ConversationalGolden*, generates a realistic opening user message, sends it to your application via the *model_callback* (see below), receives the assistant's reply, and then generates the next user message based on everything said so far. This back-and-forth loop continues until the conversation either reaches the expected outcome defined in the golden or hits a configured turn limit. The result is a completely generated conversation that is different every time it runs, yet always grounded in the same scenario constraints.

By default, DeepEval uses OpenAI models for user conversation simulation. To use Bedrock instead, the following code creates the corresponding custom DeepEval model based on Amazon Nova Micro for efficiency.

In [ ]:
import json
import boto3
from deepeval.models import DeepEvalBaseLLM


class BedrockNovaMicro(DeepEvalBaseLLM):
    """DeepEval-compatible wrapper around Amazon Nova Micro via Bedrock.
    Used as the user simulator model (cost-efficient for generating user turns) instead of the default OpenAI backend."""

    MODEL_ID = "amazon.nova-micro-v1:0"

    def __init__(self, region: str = "us-east-1"):
        self.region = region
        super().__init__(model=self.MODEL_ID)

    def load_model(self):
        return boto3.client("bedrock-runtime", region_name=self.region)

    def get_model_name(self) -> str:
        return self.MODEL_ID

    @staticmethod
    def _flatten_values(obj):
        """Recursively convert non-string values in dicts to JSON strings."""
        if isinstance(obj, dict):
            for k, v in obj.items():
                if isinstance(v, dict):
                    obj[k] = json.dumps(v)
                elif isinstance(v, list):
                    obj[k] = [json.dumps(i) if not isinstance(i, str) else i for i in v]
        elif isinstance(obj, list):
            for item in obj:
                if isinstance(item, dict):
                    BedrockNovaMicro._flatten_values(item)

    def _invoke(self, prompt: str, schema=None) -> str:
        system_text = "You are a helpful assistant. Respond only in valid JSON." if schema else None
        if schema:
            json_schema = schema.model_json_schema()
            prompt = (
                f"{prompt}\n\n"
                f"Return ONLY valid JSON matching this schema (no markdown, no extra text):\n"
                f"{json.dumps(json_schema, indent=2)}"
            )

        messages = [{"role": "user", "content": [{"text": prompt}]}]
        body = {"messages": messages, "inferenceConfig": {"maxTokens": 4096, "temperature": 0.0}}
        if system_text:
            body["system"] = [{"text": system_text}]

        response = self.model.converse(modelId=self.MODEL_ID, **body)
        text = response["output"]["message"]["content"][0]["text"]

        if schema:
            try:
                return schema.model_validate_json(text)
            except Exception:
                data = json.loads(text)
                self._flatten_values(data)
                return schema.model_validate(data)
        return text

    def generate(self, prompt: str, schema=None) -> str:
        return self._invoke(prompt, schema)

    async def a_generate(self, prompt: str, schema=None) -> str:
        return self._invoke(prompt, schema)


# Instantiate the Bedrock model for use as the user simulator
nova_micro = BedrockNovaMicro()
print(f"Model loaded: {nova_micro.get_model_name()}")

A response callback, `agent_callback`, is used by simulator for producing application responses. It wraps your conversational assistant and returns valid Turns from the obtained responses.

In [ ]:
from deepeval.test_case import ToolCall, Turn


def agent_callback(input: str) -> Turn:
    """Invoke the Strands agent and return a DeepEval Turn. 
       We use a local agent running in the notebook, however, in a real-world scenario you 
       would typically invoke a remote agent API for agent deployed to production runtime, 
       such as the one provided by AgentCore runtime running the agent service."""

    response = agent(input)
    text = response.message["content"][0]["text"] if response.message.get("content") else str(response)
    tools = [ToolCall(name=tm.tool["name"], 
                      input_parameters=tm.tool.get("input")) 
             for tm in response.metrics.tool_metrics.values()] or None
    return Turn(role="assistant", 
                content=text,
                tools_called=tools)

Next, we create the `ConversationSimulator` instance, wiring it to our agent callback and the Nova Micro model for user message generation.

In [ ]:
from deepeval.simulator import ConversationSimulator

# Create the simulator: generates user messages and feeds them to agent_callback
simulator = ConversationSimulator(model_callback=agent_callback, 
                                  simulator_model=nova_micro,
                                  async_mode=False)

The simulator uses Amazon Nova Micro model for user simulation generation and cannot exceed 10 turns during the simulation. For production workloads, increase this number to 20-30. 
Now let's start the simulation.

In [ ]:
# Run the simulation: generate synthetic conversations for each golden scenario
n_goldens = len(goldens)
test_cases = simulator.simulate(
    conversational_goldens=goldens,
    max_user_simulations=10, # max turns per conversation
    on_simulation_complete=lambda tc, i: print(f"Finished test case #{i+1} out of {n_goldens} test cases"),
)

*ConversationalTestCase* is the output of that simulation and the input to evaluation. It packages the full conversation, every user message and assistant response in order, into a structured object that DeepEval's multi-turn metrics can score. It preserves the sequential relationship between turns, so metrics can assess not just individual reply quality but also consistency, context retention, and conversation flow across the entire session.

The following code prints a preview of each simulated conversation, showing the role and a truncated content snippet for every turn.

In [ ]:
# Display a summary of each simulated conversation
print(f"\nSimulated {len(test_cases)} conversations.")
for i, conv in enumerate(test_cases):
    print(f"\n--- Conversation {i+1} ({len(conv.turns)} turns) ---")
    for turn in conv.turns:
        preview = turn.content[:120].replace("\n", " ")
        print(f" [{turn.role}] {preview}..." if len(turn.content) > 120 else f" [{turn.role}] {preview}")

## Evaluation: Custom Binary Metrics, One Per Failure Mode

Before writing metrics, we list the failure modes we want to probe for the travel booking assistant. These are the same domain failure modes we target in notebook 05 with Strands Evals. The evaluation target is the same, only the framework changes.

| # | Failure Mode | Metric Type |
|---|--------------|-------------|
| F1 | Agent **invents a flight** not present in search results | `ConversationalGEval` (binary) |
| F2 | Agent **books without confirming** passenger name / date | `ConversationalGEval` (binary) |
| F3 | Agent **forgets earlier state** (passenger name, destination) across many turns | `ConversationalGEval` (binary) |
| F5 | Agent **cancels the wrong booking** | `ConversationalGEval` (binary) |
| F6 | Agent **responds to an off-topic request** | `ConversationalGEval` (binary) |
| Composite | The assistant first meets the user's need, *then* keeps a professional tone | `ConversationalDAGMetric` (tone only evaluated if the task succeeded) |

Each `ConversationalGEval` is phrased as a single yes/no question so the verdict is binary. The LLM judge still produces a score, but because the criterion is binary, that score collapses to PASS (>= threshold) or FAIL (below threshold).

The cell below sets up the imports and a helper to make each metric instance easy to inspect.

In [ ]:
from deepeval.metrics import ConversationalGEval
from deepeval.metrics import ConversationalDAGMetric
from deepeval.metrics.dag import DeepAcyclicGraph
from deepeval.metrics.conversational_dag import (
    ConversationalTaskNode,
    ConversationalBinaryJudgementNode,
    ConversationalNonBinaryJudgementNode,
    ConversationalVerdictNode,
)
from deepeval.test_case import TurnParams
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig

# Judge model: Claude Sonnet 4.5 for strong binary judgement on domain questions.
# We reuse the BedrockNovaMicro wrapper pattern to build a Claude Sonnet 4.5 judge.
import boto3, json as _json
from deepeval.models import DeepEvalBaseLLM


class BedrockClaudeSonnet45(DeepEvalBaseLLM):
    """DeepEval-compatible wrapper around Claude Sonnet 4.5 via Bedrock, used as the evaluation judge."""

    MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

    def __init__(self, region: str = 'us-east-1'):
        self.region = region
        super().__init__(model=self.MODEL_ID)

    def load_model(self):
        return boto3.client('bedrock-runtime', region_name=self.region)

    def get_model_name(self) -> str:
        return self.MODEL_ID

    @staticmethod
    def _extract_json(text: str) -> str:
        """Strip markdown fences and extract the first {...} or [...] JSON block.

        Chooses array-first when '[' appears before '{' in the text, so array
        responses aren't mistakenly narrowed to their first object.
        """
        import re as _re
        # Remove ```json ... ``` or ``` ... ``` fences
        fenced = _re.search(r'```(?:json)?\s*(.*?)```', text, _re.DOTALL)
        if fenced:
            text = fenced.group(1)
        text = text.strip()
        first_obj = text.find('{')
        first_arr = text.find('[')
        if first_arr == -1 and first_obj == -1:
            return text
        if first_arr == -1 or (first_obj != -1 and first_obj < first_arr):
            order = [('{', '}'), ('[', ']')]
        else:
            order = [('[', ']'), ('{', '}')]
        for open_ch, close_ch in order:
            start = text.find(open_ch)
            if start == -1:
                continue
            depth = 0
            in_str = False
            escape = False
            for i in range(start, len(text)):
                ch = text[i]
                if escape:
                    escape = False
                    continue
                if ch == '\\':
                    escape = True
                    continue
                if ch == '"':
                    in_str = not in_str
                    continue
                if in_str:
                    continue
                if ch == open_ch:
                    depth += 1
                elif ch == close_ch:
                    depth -= 1
                    if depth == 0:
                        return text[start:i + 1]
        return text  # fall back to the whole text

    def _invoke(self, prompt: str, schema=None) -> str:
        system_text = 'You are a helpful assistant. Respond only in valid JSON.' if schema else None
        if schema:
            json_schema = schema.model_json_schema()
            prompt = (
                f'{prompt}\n\n'
                f'Return ONLY valid JSON matching this schema. '
                f'No markdown fences, no commentary, no prose. JSON only.\n'
                f'{_json.dumps(json_schema, indent=2)}'
            )
        messages = [{'role': 'user', 'content': [{'text': prompt}]}]
        body = {'messages': messages, 'inferenceConfig': {'maxTokens': 8192, 'temperature': 0.0}}
        if system_text:
            body['system'] = [{'text': system_text}]
        response = self.model.converse(modelId=self.MODEL_ID, **body)
        text = response['output']['message']['content'][0]['text']
        if schema:
            # Robust: strip markdown fences and extract JSON block before decoding.
            cleaned = self._extract_json(text)
            try:
                return schema.model_validate_json(cleaned)
            except Exception:
                data = _json.loads(cleaned)
                return schema.model_validate(data)
        return text

    def generate(self, prompt: str, schema=None) -> str:
        return self._invoke(prompt, schema)

    async def a_generate(self, prompt: str, schema=None) -> str:
        return self._invoke(prompt, schema)


judge_model = BedrockClaudeSonnet45()
print(f'Judge model loaded: {judge_model.get_model_name()}')

### Binary `ConversationalGEval` Metrics

Each metric below is a single `ConversationalGEval` whose `criteria` is phrased as a yes/no question. The `threshold` is set to 1.0 so anything short of a clean PASS counts as FAIL. This is how we enforce strict binary semantics on top of DeepEval's scoring mechanism.

In [ ]:
# F1: no invented flights
metric_f1_no_invented = ConversationalGEval(
    name='F1_no_invented_flights',
    criteria=(
        'Binary check: does the assistant mention or book any flight number or airline that '
        'was NOT returned by the search_flights tool earlier in the conversation? '
        'Return PASS (score 1.0) if every flight referenced came from search_flights output. '
        'Return FAIL (score 0.0) if the assistant referenced or booked any flight not present in the search results.'
    ),
    model=judge_model,
    threshold=1.0, # strict: any sub-PASS score counts as FAIL
)

# F2: confirmed before booking
metric_f2_confirmed = ConversationalGEval(
    name='F2_confirmed_before_booking',
    criteria=(
        'Binary check: consider a "booking" to have happened if the assistant either (a) called a '
        'booking tool, or (b) produced a message containing a booking confirmation (e.g. a reference '
        'like FLT-XXXX or HTL-XXXX, or the words "booked"/"confirmed"). '
        'If a booking happened in this conversation, did the assistant explicitly ask the user to '
        'confirm passenger/guest name and travel dates, and receive an acknowledgement, BEFORE the '
        'booking was produced? '
        'Return PASS (score 1.0) if confirmation happened before every booking, OR if no booking happened. '
        'Return FAIL (score 0.0) if any booking was produced without explicit prior confirmation from the user.'
    ),
    model=judge_model,
    threshold=1.0,
)

# F3: remembers earlier state
metric_f3_retention = ConversationalGEval(
    name='F3_remembers_state',
    criteria=(
        'Binary check: did the assistant ever ask the user to repeat a fact they had already provided earlier '
        '(name, destination, travel date, or booking reference)? '
        'Return PASS (score 1.0) if the assistant carried forward all earlier-provided facts without asking again. '
        'Return FAIL (score 0.0) if the assistant forgot any earlier-provided fact.'
    ),
    model=judge_model,
    threshold=1.0,
)

# F5: cancelled the correct booking
metric_f5_correct_cancel = ConversationalGEval(
    name='F5_correct_cancellation',
    criteria=(
        'Binary check: if the user asked to cancel a specific booking, did the assistant cancel exactly that '
        'booking reference and leave all other bookings intact? '
        'Return PASS (score 1.0) if the cancellation matched the user request exactly, OR if no cancellation was requested. '
        'Return FAIL (score 0.0) if the assistant cancelled the wrong booking, a non-existent reference, or '
        'cancelled multiple bookings when only one was requested.'
    ),
    model=judge_model,
    threshold=1.0,
)

# F6: did not drift off-topic
metric_f6_on_topic = ConversationalGEval(
    name='F6_on_topic',
    criteria=(
        'Binary check: did the assistant answer any off-topic request (coding help, medical/legal advice, '
        'general knowledge questions unrelated to travel)? '
        'Return PASS (score 1.0) if the assistant stayed within travel booking assistance and politely '
        'redirected any off-topic request. '
        'Return FAIL (score 0.0) if the assistant produced any off-topic content.'
    ),
    model=judge_model,
    threshold=1.0,
)

custom_binary_metrics = [
    metric_f1_no_invented,
    metric_f2_confirmed,
    metric_f3_retention,
    metric_f5_correct_cancel,
    metric_f6_on_topic,
]

print(f'Configured {len(custom_binary_metrics)} binary ConversationalGEval metrics.')

### Run the Binary Metrics and Read the Pass-Rate-Per-Failure-Mode Table

We score the simulated conversations from earlier in the notebook (`test_cases`) against the five binary metrics. Rather than averaging numeric scores, we count PASS vs FAIL directly.

In [ ]:
# Run the custom binary metrics on the simulated conversations.
results = evaluate(
    test_cases=test_cases,
    metrics=custom_binary_metrics,
    async_config=AsyncConfig(run_async=False),
)

# Build a PASS/FAIL matrix from the per-testcase results. The `results` object returned by
# evaluate() contains test_results, each of which has metrics_data entries with per-case
# score/success/threshold values. That's the authoritative source for reporting.
print()
print(f'{"Case":35s} | ' + ' | '.join(f'{m.name:30s}' for m in custom_binary_metrics))
print('-' * (35 + 3 + (30 + 3) * len(custom_binary_metrics)))

pass_counts = {m.name: 0 for m in custom_binary_metrics}
fail_counts = {m.name: 0 for m in custom_binary_metrics}

for tr in results.test_results:
    case_name = tr.name or 'Case'
    row_cells = []
    for m in custom_binary_metrics:
        # DeepEval appends " [Conversational GEval]" to the metric name; match by prefix.
        md_entry = next((md for md in tr.metrics_data if md.name.startswith(m.name)), None)
        if md_entry is None:
            row_cells.append('?')
            continue
        verdict = 'PASS' if (md_entry.success and md_entry.score >= m.threshold) else 'FAIL'
        row_cells.append(verdict)
        if verdict == 'PASS':
            pass_counts[m.name] += 1
        else:
            fail_counts[m.name] += 1
    print(f'{case_name[:35]:35s} | ' + ' | '.join(f'{v:30s}' for v in row_cells))

# Aggregate pass rate per metric across the whole run.
print()
print('Pass rate per failure mode:')
for m in custom_binary_metrics:
    p = pass_counts[m.name]
    total = p + fail_counts[m.name]
    rate = p / total if total else 0.0
    print(f' {m.name:35s} {rate:6.0%} ({p}/{total} cases)')

### Composite Check With `ConversationalDAGMetric`

Some criteria only make sense in a specific order. For example, there is no point evaluating tone when the agent did not complete the user's task. A polite agent that fails to book your flight is still a failure. `ConversationalDAGMetric` expresses this as a decision tree: check the task first, only evaluate tone if the task succeeded.

The DAG below:

1. **TaskNode** (root): summarize the conversation.
2. **Binary**: did the assistant satisfy the user's requests? No → score 0 (fail). Yes → continue.
3. **Non-binary**: classify the tone as Rude (0), Playful (5), or Professional (10).

We only award full credit when *both* the task was satisfied *and* the tone was Professional.

In [ ]:
def make_task_then_tone_metric():
    """Decision tree: task-completion gates tone evaluation, then tone is a 3-way category."""
    tone_node = ConversationalNonBinaryJudgementNode(
        criteria='Classify the assistant\'s overall tone toward the user.',
        children=[
            ConversationalVerdictNode(verdict='Rude', score=0),
            ConversationalVerdictNode(verdict='Playful', score=5),
            ConversationalVerdictNode(verdict='Professional', score=10),
        ],
    )
    task_gate = ConversationalBinaryJudgementNode(
        criteria='Did the assistant satisfy all of the user\'s requests in this conversation?',
        children=[
            ConversationalVerdictNode(verdict=False, score=0),
            ConversationalVerdictNode(verdict=True, child=tone_node),
        ],
    )
    root = ConversationalTaskNode(
        instructions='Summarize the conversation briefly before evaluating the branches below.',
        output_label='Summary',
        evaluation_params=[TurnParams.ROLE, TurnParams.CONTENT],
        children=[task_gate],
    )
    dag = DeepAcyclicGraph(root_nodes=[root])
    return ConversationalDAGMetric(name='TaskThenTone', model=judge_model, dag=dag)


# Workaround for DeepEval's stale turn_window state between test cases.
import deepeval.metrics.conversational_dag.nodes as _dag_nodes

_orig_task_execute = _dag_nodes.ConversationalTaskNode._execute
_orig_binary_execute = _dag_nodes.ConversationalBinaryJudgementNode._execute
_orig_nonbinary_execute = _dag_nodes.ConversationalNonBinaryJudgementNode._execute

def _patched_execute(orig_fn):
    def wrapper(self, *args, **kwargs):
        self.turn_window = None
        return orig_fn(self, *args, **kwargs)
    return wrapper

_dag_nodes.ConversationalTaskNode._execute = _patched_execute(_orig_task_execute)
_dag_nodes.ConversationalBinaryJudgementNode._execute = _patched_execute(_orig_binary_execute)
_dag_nodes.ConversationalNonBinaryJudgementNode._execute = _patched_execute(_orig_nonbinary_execute)

# Run the composite metric
evaluate(
    test_cases=test_cases,
    metrics=[make_task_then_tone_metric()],
    async_config=AsyncConfig(run_async=False),
)

# Restore originals
_dag_nodes.ConversationalTaskNode._execute = _orig_task_execute
_dag_nodes.ConversationalBinaryJudgementNode._execute = _orig_binary_execute
_dag_nodes.ConversationalNonBinaryJudgementNode._execute = _orig_nonbinary_execute

### Interpreting the Results

Read the results as **pass rate per failure mode**:

- If `F1_no_invented_flights` drops below 100% → the agent is hallucinating flights. Fix the prompt to constrain grounding, or add a verification step after tool calls.
- If `F2_confirmed_before_booking` drops → the prompt's confirmation instruction is not being respected. Tighten the wording or add a guard in the tool call layer.
- If `F3_remembers_state` drops on long conversations → context window handling or history summarization is not carrying facts forward correctly.
- If the composite `TaskThenTone` scores cluster at 0 → the task gate failed, so tone wasn't even evaluated. Fix the task-level failures first before worrying about tone.

Because every metric is binary and tied to a specific, named failure mode, any regression is actionable without having to compare numeric Likert scores or debate whether a 0.5 is better than a 0.75.

## Best Practices for Multi-Turn Evaluation Metrics

| Practice | Guidance |
|----------|----------|
| Keep each metric's criterion to **one binary question** | If a criterion tests more than one thing, split it into two metrics so a FAIL points to exactly one failure mode. |
| Use `threshold=1.0` on `ConversationalGEval` to enforce strict binary | A partial-credit score on a binary question is almost always a FAIL in practice; treat it that way. |
| Derive metrics from **observed** failures | Start from your top failure hypotheses or from incidents you've seen. Don't pre-emptively write metrics for hypothetical problems. |
| Validate the judge against human labels periodically | The same judge-calibration step shown in notebook 05 applies here. Expect to revise rubrics once you see where the judge and humans disagree. |
| Grow the scenario set as you find new failures | Every incident or support ticket is an opportunity to add a case + metric and make it a permanent regression check. |

## Next Steps

- Continue to **`04-11-05-strands-evaluators.ipynb`** to see the same binary-custom-evaluator pattern using Strands Evals.
- Or skip to **`04-11-08-e2e-pipeline.ipynb`** to wire dimensions + simulation + custom binary evaluators into a full pipeline.